# **Desafio Estatística com Python**
# Frequência e Medidas

### Squad Nina da Hora | Bootcamp Data Analytics 2026.1

## **Configurações Iniciais**

Fonte dos dados: [Kaggle - Netflix Shows and Movies - Exploratory Analysis](https://www.kaggle.com/code/shivamb/netflix-shows-and-movies-exploratory-analysis)

In [ ]:
# Importa as bibliotecas
import pandas as pd
import matplotlib.pyplot as plt

# Carregamento da base de dados sobre catalogo da Netflix via Google Drive
# Armazena o id do arquivo csv e o insere na variavel da url
sheet_id = '1OJk-yF4ZgqyKP9Q6qgfWziiOQEOCV7mML0p0_AjdWGI'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv'

# Le o csv e armazena os dados em um data frame
df = pd.read_csv(url)
df.head(0)

*Variáveis:*
* `show_id` - id único do filme/série.
* `title` - título do filme ou série
* `director` - diretor do filme ou série
* `cast` - elenco do filme ou série
* `country` - país do filme ou série
* `date_added` - data que foi adicionado no Netflix
* `release_year` - ano de lançamento original do filme
* `rating` - classificação da televisão
* `duration` - duração total do filme ou série.
* `listed_in` - categoria ou gênero do filme ou série.
* `description` - descrição do filme ou série.
* `type` - tipo de filme ou série

## **Exploração Inicial**

*   Quantas linhas e colunas tem o dataset?
*   Quais são os tipos das variáveis e se há valores ausentes?

In [ ]:
# Quantas linhas e colunas tem o dataset?
linhas = df.shape[0]
colunas = df.shape[1]
print(f"O dataset possui {linhas} linhas e {colunas} colunas.")

In [ ]:
# Quais são os tipos das variáveis?
df.info()

In [ ]:
# Há valores ausentes no dataset?
# Verificando a existência de valores ausentes em cada coluna do dataset
df.isna().sum()

In [ ]:
# Exibindo tabela com a quantidade de valores ausentes em cada coluna em que ocorrem, e a proporção destes
ausentes = df.isna().sum()
porcentagem = 100 * ausentes / len(df)
aus_table = pd.concat([ausentes, porcentagem], axis=1)
aus_table = aus_table.rename(columns= {0: '# Count: valores nulos', 1 : '% do Total'})
aus_table = aus_table[aus_table.iloc[:,1] != 0].sort_values(
    '% do Total', ascending=False).round(1)
print("O dataframe contém " + str(df.shape[1]) + " colunas.\n"
                                  "Contendo " + str(aus_table.shape[0]) +
                                  " colunas com valores faltantes.")
aus_table

## **Análises de Frequência**

* Qual a proporção de filmes vs. séries no catálogo?
* Qual o gênero mais frequente?

In [ ]:
# Analisa a proporcao de cada tipo de midia (filme ou serie)
total_tipo_midia = df['type'].value_counts() # contagem total de cada tipo de midia
total_tipo_midia_perc = df['type'].value_counts(normalize=True).round(2) * 100 # contagem percentual

# Junta o total e a proporcao de cada tipo de midia num unico df
df_tipo_midia= pd.concat([total_tipo_midia, total_tipo_midia_perc], axis=1)

# Renomeia as colunas
df_tipo_midia.index.name = 'Tipo de mídia'
df_tipo_midia.rename(columns={
    'count': 'Total',
    'proportion': 'Total em %'
  }, inplace=True)

# Imprime resposta
filmes_perc = df_tipo_midia['Total em %'].loc['Movie']
series_perc = df_tipo_midia['Total em %'].loc['TV Show']
print(f'O catálogo possui {filmes_perc}% de filmes e {series_perc}% de séries (razão: {filmes_perc / series_perc:.2f}/1).')

df_tipo_midia

In [ ]:
"""
Funcao que calcula a frequencia absoluta e percentual de uma Series e retorna um DataFrame.
"""
def calcular_frequencia(tabela, idx_txt):
  # Conta a quantidade absoluta de cada elemento
  quantidade = tabela.value_counts().rename('Total')

  # Calcula as porcentagens, arrendondando para 2 casas
  porporcao = tabela.value_counts(normalize=True).rename('Total em %')
  porporcao = (porporcao * 100).round(2)

  # Junta as tabelas das contagens
  df_resultado = pd.concat([quantidade, porporcao], axis=1)
  df_resultado.index.name = idx_txt # Renomeia o index

  return df_resultado

# Armazena o nome da coluna
idx_genero = 'Gênero'

# Extrai cada genero listado em ´listed_in´ para novas linhas
generos_separados = df['listed_in'].str.split(',').explode()

In [ ]:
# Gera o df com as contagens dos generos
df_generos = calcular_frequencia(generos_separados, idx_genero)

# Armazena o genero mais frequente
genero_moda = generos_separados.mode()[0]
# Imprime resposta
print(f'O gênero mais frequente é "{genero_moda}", presente em {df_generos["Total em %"].loc[genero_moda]}% do catálogo.')

df_generos.head()

### Adicional - Limpeza e Tratamento dos Dados de Gênero
O tipo de mídia esta presente em alguns genêros, o que pode distorcer a análise.

In [ ]:
# Atribui padrao aos elementos que tem apenas o tipo de midia
sem_genero = 'Sem Gênero'
# Dicionario mapeando os generos do catalogo para unificacao
mapeamento_genero = {
  'Movies': sem_genero,
  'TV Shows': sem_genero,
  'Docuseries': 'Documentaries',
  'Stand-Up Comedy & Talk Shows': 'Stand-Up Comedy,Talk Shows',
  'Classic & Cult TV': 'Classic,Cult',
}

# Substitui e separa os generos em novas linhas
df_generos_limpos = generos_separados.replace(mapeamento_genero).str.split(',').explode()

# Cria um dicionario para remover o tipo de midia dos generos
conteudo_limpeza = {}
# Adiciona os plurais dos tipos de midia
for tipo in df_tipo_midia.index:
  conteudo_limpeza[tipo + 's'] = ''

# Cria uma lista para remover termos redundantes e adiciona ao dicionario
termos_remover = ['TV', 'Series', 'Features']
for palavra in termos_remover:
  conteudo_limpeza[palavra] = ''

# Faz a limpeza dos dados, seguindo o dic e retirando espacos
df_generos_limpos = df_generos_limpos.replace(conteudo_limpeza, regex=True).str.strip()

In [ ]:
# Armazena o genero mais frequente - precisa vir antes da funcao
moda_generos_limpos = df_generos_limpos.mode()[0]

# Gera o df com as contagens dos generos tratados
df_generos_limpos = calcular_frequencia(df_generos_limpos, idx_genero)

# Imprime resposta
perc_moda_generos_limpos = df_generos_limpos['Total em %'].loc[moda_generos_limpos]
print(f'Análise Tratada: Após a limpeza, o gênero mais frequente é "{moda_generos_limpos}", representando {perc_moda_generos_limpos}% do catálogo.')

df_generos_limpos.head()

## **Análises Estatísticas**

* Qual a média, mediana e moda do tempo de duração dos
filmes?
* Qual o filme mais curto e mais longo?

In [ ]:
# 1. Filtrando apenas os filmes e limpando a coluna de duração
df_filmes = df[df["type"] == "Movie"].copy()
df_filmes["duration_num"] = df_filmes["duration"].str.replace(" min", "").astype(float)

# 2. Criando a agregação única para as estatísticas (Média, Mediana e Moda)
tabela_medidas = df_filmes["duration_num"].agg(["mean", "median"])
tabela_medidas["mode"] = df_filmes["duration_num"].mode().iloc[0]

# Formatando a tabela para exibição elegante
tabela_summary = pd.DataFrame(tabela_medidas).T
tabela_summary.columns = ["Média", "Mediana", "Moda"]
tabela_summary.index = ["Duração (Minutos)"]

print("--- ESTATÍSTICAS DE TENDÊNCIA CENTRAL ---")
display(tabela_summary)

# 3. Identificando o filme mais curto e o mais longo
linha_mais_curto = df_filmes["duration_num"].idxmin()
linha_mais_longo = df_filmes["duration_num"].idxmax()

filme_mais_curto = df_filmes.loc[linha_mais_curto]
filme_mais_longo = df_filmes.loc[linha_mais_longo]

print("\n--- EXTREMOS DO CATÁLOGO ---")
print(f"Filme Mais Curto: \"{filme_mais_curto['title']}\" com {filme_mais_curto['duration_num']:.0f} minuto(s).")
print(f"Filme Mais Longo: \"{filme_mais_longo['title']}\" com {filme_mais_longo['duration_num']:.0f} minutos.")

## **Visualização de Dados**

* Criar um gráfico de barras para mostrar a quantidade de títulos por gênero.
* Criar um histograma para analisar a distribuição da duração dos
filmes.

In [ ]:
# Importar a biblioteca para gerar os gráficos

import seaborn as sns

# Variável criada para facilitar
col_genero = 'listed_in'

# Preservando o df original, para evitar alterações indesejadas
# O str.split() usado para dividir os gêneros listados na coluna
# 'listed_in' em listas de gêneros separados por vírgula, fazendo com que seja guardado na
# nova coluna 'col_genero' do DataFrame 'df_separado'.

df_separado = df.copy()
df_separado[col_genero] = df_separado[col_genero].str.split(', ')

# O .explode() é usado para expandir as listas que estão dentro de uma única linha, transformando em várias linhas
# O .nunique() é utilizado para fazer a contagem dos valores únicos presentes na coluna em questão.
df_por_gen = df_separado.explode(col_genero)
total_gen_unicos = df_por_gen[col_genero].nunique()
print(f"O número total de gêneros únicos no catálogo é: {total_gen_unicos}")

# Contando exatamente a quantidade de títulos por gênero.

cont_generos = df_por_gen[col_genero].value_counts()

# Criando o gráfico de barras
plt.figure(figsize=(12, 8))
sns.set_theme(style="whitegrid")

top_genero = cont_generos.head(15) # Isola os 15 primeiros gêneros. 

# O sns.barplot() usado para criar o gráfico de barras.

ax = sns.barplot(
        x=top_genero.values,            # Quantidade de títulos 
        y=top_genero.index,             # Nome dos gêneros
        hue =top_genero.index,          # Usa diferente cores para cada barra
        palette="magma",                # Paleta de cores
        legend=False                    # Desativa a caixa de legendas.
)

# O for é usado aqui, para que em cada barra fosse adicionado a quantidade de títulos.

for container in ax.containers:
    ax.bar_label(container, fmt='%d', padding=5, fontsize=10, color='black', fontweight='bold')

# Criação do título do gráfico e as legendas de cada eixo.

plt.title('Top 15 gêneros mais frequentes no catálogo da Netflix', fontsize=16, fontweight='bold',
          pad=15)
plt.xlabel('Quantidade de títulos', fontsize=12)
plt.ylabel('Gêneros', fontsize=12)


plt.tight_layout()  # Ajusta os espaços no gráfico.
plt.show()          # Para exibir o ráfico.

In [ ]:
# Gráfico de barras usando a base limpa e tratada dos gêneros
top_genero_limpo = df_generos_limpos.head(15)
print(top_genero_limpo)

# Criando o gráfico

plt.figure(figsize=(12, 8))
sns.set_theme(style="whitegrid")

ax = sns.barplot(
        x=top_genero_limpo['Total'],
        y=top_genero_limpo.index,
        hue =top_genero_limpo.index,
        palette="magma",
        legend=False
)

for container in ax.containers:
    ax.bar_label(container, fmt='%d', padding=5, fontsize=10, color='black', fontweight='bold')


plt.title('Top 15 gêneros mais frequentes no catálogo da Netflix (Limpo)', fontsize=16, fontweight='bold',
            pad=15)
plt.xlabel('Quantidade de títulos', fontsize=12)
plt.ylabel('Gêneros', fontsize=12)

In [ ]:
# Criando um histograma para visualizar a distribuição da duração dos filmes

df_filmes = df[df['type'] == 'Movie'].copy()

# Para fazer uma limpeza na coluna em questão.

df_filmes['duration'] = df_filmes['duration'].str.replace(' min', '').astype(int)

plt.figure(figsize=(10, 6))
sns.histplot(df_filmes['duration'], bins=30, kde=True, color='skyblue')
plt.title('Distribuição da Duração dos Filmes no Catálogo da Netflix', fontsize=16, fontweight='bold',
     pad=15)
plt.xlabel('Duração (minutos)', fontsize=12)
plt.ylabel('Frequência', fontsize=12)
plt.tight_layout()
plt.show()



## **Atividade Extra**

* Quais são os 5 países que possuem mais produções no catálogo?




In [ ]:
# Identificando os 5 países com maior número de produções no catálogo
# Alguns títulos possuem mais de um país associado, por isso os países
# são separados e analisados individualmente

paises = df['country'].dropna().str.split(', ')
paises_explodidos = paises.explode()

top5_paises = (
    paises_explodidos
    .value_counts()
    .head(5)
    .reset_index()
)

top5_paises.columns = ['País', 'Quantidade']

top5_paises.insert(
    0,
    'Ranking',
    range(1, len(top5_paises) + 1)
)

top5_paises

In [ ]:
# Visualizando os 5 países com mais produções no catálogo

plt.figure(figsize=(8,5))

ax = plt.bar(
    top5_paises['País'],
    top5_paises['Quantidade'],
    color='pink'
)

plt.title('Top 5 países com mais produções')
plt.xlabel('País')
plt.ylabel('Quantidade')
plt.xticks(rotation=45)

# Exibindo os valores sobre as barras
for bar in ax:
    y = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        y,
        int(y),
        ha='center',
        va='bottom'
    )

plt.show()